In [1]:
import pandas as pd
import numpy as np
import os

In [ ]:
# --- Configuration ---
FILE_NET_GRID_DRAW = 'Lastgang Mainkofen_2023.xlsx'
FILE_ELEC_BIOMASS_DRAW = 'biomass_electricity_hourly_2023'
DIRECTORY = '../timeseries_data'
FILE_GROSS_DEMAND = os.path.join(DIRECTORY, 'biomass_electricity_prod.csv')

In [3]:
# Inputs from your model:
BIOMASS_ANNUAL_FUEL_KWH = 23934060
BIOMASS_EFFICIENCY_EL = 0.18

# Biomass OFF period (May 1st to Sept 14th inclusive)
SUMMER_START = '05-01'
SUMMER_END = '09-14'
START_DATE = '2023-01-01 00:00:00'
END_DATE = '2023-12-31 23:00:00'


In [4]:
# --- 1. Load and Clean Net Grid Draw Data ---
df_raw = pd.read_excel(FILE_NET_GRID_DRAW)
COL_POWER = 'Wirkleistung [kW]/electricity consumption'
df_raw['DT-Index'] = pd.to_datetime(df_raw['DT-Index'])
df_raw = df_raw.set_index('DT-Index')

# Resample from 15-minute kW (power) to 1-hour kWh (energy)
df_hourly = df_raw[COL_POWER].resample('H').sum() * 0.25
df_net_grid_draw = df_hourly.rename('net_grid_draw_kwh')
df_net_grid_draw

C:\Users\dnouicer.THDITZ0329\AppData\Local\Temp\ipykernel_20376\3412127003.py:8: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_hourly = df_raw[COL_POWER].resample('H').sum() * 0.25


DT-Index
2023-01-01 00:00:00    493.490
2023-01-01 01:00:00    640.655
2023-01-01 02:00:00    640.270
2023-01-01 03:00:00    645.720
2023-01-01 04:00:00    660.370
                        ...   
2023-12-31 20:00:00    722.540
2023-12-31 21:00:00    707.045
2023-12-31 22:00:00    690.675
2023-12-31 23:00:00    681.800
2024-01-01 00:00:00    170.985
Freq: h, Name: net_grid_draw_kwh, Length: 8761, dtype: float64

In [5]:
# --- 2. Calculate Biomass Electricity Production Time Series ---

index_full_year = pd.date_range(start=START_DATE, end=END_DATE, freq='H')
df_biomass_prod = pd.DataFrame(index=index_full_year)
df_biomass_prod['biomass_el_prod_kwh'] = 0.0 # Start with 0 production
df_biomass_prod

C:\Users\dnouicer.THDITZ0329\AppData\Local\Temp\ipykernel_20376\4207603236.py:3: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  index_full_year = pd.date_range(start=START_DATE, end=END_DATE, freq='H')


,biomass_el_prod_kwh
2023-01-01 00:00:00,0.0
2023-01-01 01:00:00,0.0
2023-01-01 02:00:00,0.0
2023-01-01 03:00:00,0.0
2023-01-01 04:00:00,0.0
...,...
2023-12-31 19:00:00,0.0
2023-12-31 20:00:00,0.0
2023-12-31 21:00:00,0.0
2023-12-31 22:00:00,0.0


In [8]:
# Identify hours outside the summer shutdown period
biomass_on_condition = ~(
    (df_biomass_prod.index.strftime('%m-%d') >= SUMMER_START) &
    (df_biomass_prod.index.strftime('%m-%d') <= SUMMER_END)
)
total_on_hours = biomass_on_condition.sum()
total_annual_production = - BIOMASS_ANNUAL_FUEL_KWH * BIOMASS_EFFICIENCY_EL
hourly_rate = total_annual_production / total_on_hours
hourly_rate

-787.3046052631579

In [9]:
# Set production rate during ON hours
df_biomass_prod.loc[biomass_on_condition, 'biomass_el_prod_kwh'] = hourly_rate
df_biomass_prod

,biomass_el_prod_kwh
2023-01-01 00:00:00,-787.304605
2023-01-01 01:00:00,-787.304605
2023-01-01 02:00:00,-787.304605
2023-01-01 03:00:00,-787.304605
2023-01-01 04:00:00,-787.304605
...,...
2023-12-31 19:00:00,-787.304605
2023-12-31 20:00:00,-787.304605
2023-12-31 21:00:00,-787.304605
2023-12-31 22:00:00,-787.304605


In [21]:
# --- 3. Combine to get Gross Demand ---
df_gross_demand = pd.concat([df_net_grid_draw, df_biomass_prod], axis=1, join='outer')
df_gross_demand['total_gross_demand_kwh'] = - df_gross_demand['net_grid_draw_kwh'] - df_gross_demand['biomass_el_prod_kwh']
df_gross_demand =df_gross_demand.fillna(0)
df_gross_demand

,net_grid_draw_kwh,biomass_el_prod_kwh,total_gross_demand_kwh
2023-01-01 00:00:00,493.490,787.304605,-1280.794605
2023-01-01 01:00:00,640.655,787.304605,-1427.959605
2023-01-01 02:00:00,640.270,787.304605,-1427.574605
2023-01-01 03:00:00,645.720,787.304605,-1433.024605
2023-01-01 04:00:00,660.370,787.304605,-1447.674605
...,...,...,...
2023-12-31 20:00:00,722.540,787.304605,-1509.844605
2023-12-31 21:00:00,707.045,787.304605,-1494.349605
2023-12-31 22:00:00,690.675,787.304605,-1477.979605
2023-12-31 23:00:00,681.800,787.304605,-1469.104605


In [22]:
# Apply negative sign for Calliope demand and rename column
df_calliope_output = pd.DataFrame(
    df_gross_demand['total_gross_demand_kwh'],
    columns=['demand_electricity']
)
df_calliope_output = df_calliope_output.fillna(0) # Fill potential NaNs from resampling edge cases
df_calliope_output

,demand_electricity


In [23]:
# Create the directory and save the file
os.makedirs(DIRECTORY, exist_ok=True)
df_gross_demand.to_csv(FILE_GROSS_DEMAND, header=True, index=True, date_format='%Y-%m-%d %H:%M:%S')

In [12]:
# Create the directory and save the file
os.makedirs(DIRECTORY, exist_ok=True)
df_biomass_prod.to_csv(FILE_GROSS_DEMAND, header=True, index=True, date_format='%Y-%m-%d %H:%M:%S')